# DL600 Exercises -- Attention, Positional Encoding, and Transformer Architecture

Work through these exercises after completing the DL600 module. Try each exercise before looking at the solutions at the bottom.

**Topics covered:** Attention math, positional encoding, transformer architecture details, masking, and implementing scaled dot-product attention from scratch.

---
## Exercise 1: Scaled Dot-Product Attention -- Manual Computation

Given the following Query, Key, and Value matrices:

$$Q = \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix}, \quad K = \begin{bmatrix} 1 & 0 \\ 0 & 1 \\ 1 & 1 \end{bmatrix}, \quad V = \begin{bmatrix} 1 & 2 \\ 3 & 4 \\ 5 & 6 \end{bmatrix}$$

where $d_k = 2$ (dimension of keys).

Compute step by step:
1. $QK^T$ (the raw attention scores)
2. $\frac{QK^T}{\sqrt{d_k}}$ (scaled scores)
3. $\text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)$ (attention weights)
4. The final output: $\text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$

In [ ]:
import numpy as np

Q = np.array([[1, 0],
              [0, 1]], dtype=np.float32)

K = np.array([[1, 0],
              [0, 1],
              [1, 1]], dtype=np.float32)

V = np.array([[1, 2],
              [3, 4],
              [5, 6]], dtype=np.float32)

d_k = K.shape[1]  # 2

# Step 1: QK^T
scores = None  # TODO: Q @ K.T
print(f"Step 1 - QK^T:\n{scores}\n")

# Step 2: Scale by sqrt(d_k)
scaled_scores = None  # TODO
print(f"Step 2 - Scaled:\n{scaled_scores}\n")

# Step 3: Softmax (row-wise)
def softmax(x):
    # TODO: implement row-wise softmax
    pass

attn_weights = None  # TODO
print(f"Step 3 - Attention weights:\n{attn_weights}\n")
print(f"Row sums (should be 1): {attn_weights.sum(axis=1)}\n")

# Step 4: Weighted sum of values
output = None  # TODO: attn_weights @ V
print(f"Step 4 - Output:\n{output}")

---
## Exercise 2: Implement Scaled Dot-Product Attention from Scratch

Implement the full scaled dot-product attention function in PyTorch:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Your implementation must:
1. Handle batched inputs: `Q, K, V` of shape `(batch, seq_len, d_k)`
2. Support an optional mask to ignore certain positions
3. Return both the output and the attention weights

In [ ]:
import torch
import torch.nn.functional as F
import math

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Compute scaled dot-product attention.

    Args:
        Q: (batch, seq_len_q, d_k)
        K: (batch, seq_len_k, d_k)
        V: (batch, seq_len_k, d_v)
        mask: optional (batch, seq_len_q, seq_len_k), 0 = masked position

    Returns:
        output: (batch, seq_len_q, d_v)
        attn_weights: (batch, seq_len_q, seq_len_k)
    """
    d_k = Q.size(-1)

    # TODO: Step 1 - Compute scaled attention scores
    # scores = ...

    # TODO: Step 2 - Apply mask (set masked positions to -inf)
    # if mask is not None:
    #     scores = ...

    # TODO: Step 3 - Softmax
    # attn_weights = ...

    # TODO: Step 4 - Weighted sum of values
    # output = ...

    # return output, attn_weights
    pass


# Test
torch.manual_seed(42)
batch, seq_len, d_k = 2, 4, 8
Q = torch.randn(batch, seq_len, d_k)
K = torch.randn(batch, seq_len, d_k)
V = torch.randn(batch, seq_len, d_k)

# output, weights = scaled_dot_product_attention(Q, K, V)
# print(f"Output shape: {output.shape}")       # (2, 4, 8)
# print(f"Weights shape: {weights.shape}")      # (2, 4, 4)
# print(f"Weights sum: {weights.sum(dim=-1)}")  # Should be 1.0 for each row

---
## Exercise 3: Why Scale by sqrt(d_k)?

The scaling factor $\frac{1}{\sqrt{d_k}}$ is critical for transformer training stability.

1. If Q and K have entries drawn from N(0, 1), what is the variance of each entry in QK^T *without* scaling? (Hint: variance of a dot product of d_k independent N(0,1) pairs)

2. What happens to the softmax output when the input values are very large? (Hint: think about the gradient)

3. How does dividing by $\sqrt{d_k}$ fix this problem?

4. Verify empirically: compare the standard deviation of attention scores with and without scaling for d_k = 64, 256, 512.

In [ ]:
import torch
import math

# Empirical verification
for d_k in [64, 256, 512]:
    Q = torch.randn(100, 20, d_k)
    K = torch.randn(100, 20, d_k)
    
    # Unscaled
    scores_unscaled = torch.bmm(Q, K.transpose(1, 2))
    
    # Scaled
    scores_scaled = scores_unscaled / math.sqrt(d_k)
    
    print(f"d_k={d_k:3d} | Unscaled std: {scores_unscaled.std():.2f} "
          f"| Scaled std: {scores_scaled.std():.2f} "
          f"| sqrt(d_k): {math.sqrt(d_k):.2f}")

# TODO: Answer the conceptual questions above

---
## Exercise 4: Positional Encoding Analysis

The sinusoidal positional encoding from "Attention Is All You Need" uses:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

1. For `d_model=4`, manually compute the positional encoding vectors for positions 0, 1, and 2.
2. Compute the **dot product** between PE(0) and PE(1), and between PE(0) and PE(10). Which is larger? Why?
3. Why do the authors use both sin and cos? What property does this give the encoding?
4. What is the advantage of sinusoidal encoding over learned positional embeddings?

In [ ]:
import numpy as np

d_model = 4

def positional_encoding(pos, d_model):
    """Compute the positional encoding vector for a single position."""
    pe = np.zeros(d_model)
    for i in range(0, d_model, 2):
        denominator = 10000 ** (i / d_model)
        pe[i] = np.sin(pos / denominator)
        if i + 1 < d_model:
            pe[i + 1] = np.cos(pos / denominator)
    return pe

# Compute PE for positions 0, 1, 2
for pos in [0, 1, 2]:
    pe = positional_encoding(pos, d_model)
    print(f"PE({pos}) = {np.round(pe, 4)}")

# Dot products to measure similarity
pe0 = positional_encoding(0, 64)
pe1 = positional_encoding(1, 64)
pe10 = positional_encoding(10, 64)
pe100 = positional_encoding(100, 64)

print(f"\nDot product PE(0) . PE(1):   {np.dot(pe0, pe1):.4f}")
print(f"Dot product PE(0) . PE(10):  {np.dot(pe0, pe10):.4f}")
print(f"Dot product PE(0) . PE(100): {np.dot(pe0, pe100):.4f}")

# TODO: Answer the conceptual questions

---
## Exercise 5: Transformer Architecture -- Component Inventory

A standard Transformer encoder block contains the following components. For each, state its purpose and what would happen if you removed it.

1. **Multi-Head Attention**
2. **Residual (Skip) Connection**
3. **Layer Normalization**
4. **Feed-Forward Network (FFN)**
5. **Dropout**

Additionally:
- What is the typical size of the FFN inner dimension relative to d_model?
- Why does the FFN use two linear layers with a non-linearity in between, rather than one large linear layer?

**Your answers:**

---
## Exercise 6: Masking in Transformers

Transformers use two types of masks:

1. **Padding mask**: Prevents attention to `<PAD>` tokens
2. **Causal (look-ahead) mask**: Prevents attention to future tokens (used in decoders)

Implement both masks and visualize them.

For the padding mask, given these sequences:
- Sequence 1: [5, 12, 3, 0, 0] (length 3, padded to 5)
- Sequence 2: [8, 2, 6, 7, 0] (length 4, padded to 5)

For the causal mask, create a mask for sequence length 5.

In [ ]:
import torch
import matplotlib.pyplot as plt

# Padding mask
sequences = torch.tensor([
    [5, 12, 3, 0, 0],  # PAD_IDX = 0
    [8, 2, 6, 7, 0],
])

def create_padding_mask(sequences, pad_idx=0):
    """Create padding mask: 1 for real tokens, 0 for padding.
    
    Returns shape: (batch, 1, 1, seq_len) for broadcasting with attention scores.
    """
    # TODO
    pass

# pad_mask = create_padding_mask(sequences)
# print(f"Padding mask shape: {pad_mask.shape}")
# print(f"Mask[0]: {pad_mask[0].squeeze()}")
# print(f"Mask[1]: {pad_mask[1].squeeze()}")


# Causal mask
def create_causal_mask(seq_len):
    """Create causal (look-ahead) mask.
    
    Position i can only attend to positions <= i.
    Returns shape: (1, 1, seq_len, seq_len)
    """
    # TODO: create lower-triangular matrix of 1s
    pass

# causal_mask = create_causal_mask(5)
# print(f"\nCausal mask (5x5):")
# print(causal_mask.squeeze())


# Visualize masks
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
# ax1.imshow(pad_mask[0].squeeze().unsqueeze(0).expand(5, 5), cmap='Blues')
# ax1.set_title("Padding Mask (Seq 1)")
# ax2.imshow(causal_mask.squeeze(), cmap='Blues')
# ax2.set_title("Causal Mask")
# plt.tight_layout()
# plt.show()

---
## Exercise 7: Multi-Head Attention -- Shape Tracking

In multi-head attention with `d_model=512` and `num_heads=8`:

1. What is `d_k` (dimension per head)?
2. Given input `x` of shape `(batch=4, seq_len=20, d_model=512)`, trace the shapes through every operation:
   - After Q, K, V linear projections
   - After splitting into heads
   - After scaled dot-product attention (per head)
   - After concatenating heads
   - After the output projection
3. How many total parameters does multi-head attention have (including all linear layers)?
4. If we used a single head with `d_k=512` instead, would we have more or fewer parameters?

In [ ]:
import torch
import torch.nn as nn

batch = 4
seq_len = 20
d_model = 512
num_heads = 8
d_k = d_model // num_heads  # TODO: compute

print(f"d_k = d_model / num_heads = {d_model} / {num_heads} = {d_k}")

x = torch.randn(batch, seq_len, d_model)

# Linear projections
W_q = nn.Linear(d_model, d_model, bias=False)
W_k = nn.Linear(d_model, d_model, bias=False)
W_v = nn.Linear(d_model, d_model, bias=False)
W_o = nn.Linear(d_model, d_model, bias=False)

Q = W_q(x)  # shape: ?
K = W_k(x)  # shape: ?
V = W_v(x)  # shape: ?
print(f"\nAfter projections: Q={Q.shape}, K={K.shape}, V={V.shape}")

# Split into heads
Q_heads = Q.view(batch, seq_len, num_heads, d_k).transpose(1, 2)  # shape: ?
K_heads = K.view(batch, seq_len, num_heads, d_k).transpose(1, 2)
V_heads = V.view(batch, seq_len, num_heads, d_k).transpose(1, 2)
print(f"After split:       Q={Q_heads.shape}")

# Attention per head
import math
scores = torch.matmul(Q_heads, K_heads.transpose(-2, -1)) / math.sqrt(d_k)
attn_weights = torch.softmax(scores, dim=-1)
attn_output = torch.matmul(attn_weights, V_heads)
print(f"Attention output:   {attn_output.shape}")

# Concatenate heads
concat = attn_output.transpose(1, 2).contiguous().view(batch, seq_len, d_model)
print(f"After concat:       {concat.shape}")

# Output projection
output = W_o(concat)
print(f"Final output:       {output.shape}")

# TODO: Count parameters
total_params = sum(p.numel() for p in [W_q.weight, W_k.weight, W_v.weight, W_o.weight])
print(f"\nTotal parameters (no bias): {total_params:,}")
print(f"Formula: 4 * d_model^2 = 4 * {d_model}^2 = {4 * d_model**2:,}")

---
---
# Solutions

Expand each section below to see the solution.

<details>
<summary><b>Solution 1: Scaled Dot-Product Attention -- Manual Computation</b></summary>

```python
import numpy as np

Q = np.array([[1, 0], [0, 1]], dtype=np.float32)
K = np.array([[1, 0], [0, 1], [1, 1]], dtype=np.float32)
V = np.array([[1, 2], [3, 4], [5, 6]], dtype=np.float32)
d_k = 2

# Step 1: QK^T
scores = Q @ K.T
# [[1*1+0*0, 1*0+0*1, 1*1+0*1],   [[1, 0, 1],
#  [0*1+1*0, 0*0+1*1, 0*1+1*1]] =  [0, 1, 1]]

# Step 2: Scale
scaled = scores / np.sqrt(d_k)
# [[0.707, 0.000, 0.707],
#  [0.000, 0.707, 0.707]]

# Step 3: Softmax (row-wise)
def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

attn_weights = softmax(scaled)
# Row 0: softmax([0.707, 0, 0.707]) = [0.365, 0.180, 0.365] (approx)
# (0 gets less weight because it is lower than the other two)
# Row 1: softmax([0, 0.707, 0.707]) = [0.180, 0.365, 0.365] (approx; note exact values differ)

# Step 4: Output
output = attn_weights @ V
# Each output row is a weighted average of V rows.
# Row 0: 0.365*[1,2] + 0.180*[3,4] + 0.365*[5,6] (emphasizes keys 0 and 2)
# Row 1: 0.180*[1,2] + 0.365*[3,4] + 0.365*[5,6] (emphasizes keys 1 and 2)

print(f"Output:\n{np.round(output, 3)}")
```

**Interpretation:** Query 0 is [1,0], so it attends most to keys that have a large first component (key 0 and key 2). Query 1 is [0,1], so it attends most to keys with a large second component (key 1 and key 2). Key 2 = [1,1] has high similarity with both queries.
</details>

<details>
<summary><b>Solution 2: Implement Scaled Dot-Product Attention from Scratch</b></summary>

```python
import torch
import torch.nn.functional as F
import math

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Args:
        Q: (batch, seq_len_q, d_k)
        K: (batch, seq_len_k, d_k)
        V: (batch, seq_len_k, d_v)
        mask: optional (batch, seq_len_q, seq_len_k)
    """
    d_k = Q.size(-1)

    # Step 1: Compute scaled scores
    scores = torch.bmm(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    # scores: (batch, seq_len_q, seq_len_k)

    # Step 2: Apply mask
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))

    # Step 3: Softmax
    attn_weights = F.softmax(scores, dim=-1)

    # Step 4: Weighted sum
    output = torch.bmm(attn_weights, V)

    return output, attn_weights


# Test
torch.manual_seed(42)
batch, seq_len, d_k = 2, 4, 8
Q = torch.randn(batch, seq_len, d_k)
K = torch.randn(batch, seq_len, d_k)
V = torch.randn(batch, seq_len, d_k)

output, weights = scaled_dot_product_attention(Q, K, V)
print(f"Output shape: {output.shape}")       # (2, 4, 8)
print(f"Weights shape: {weights.shape}")      # (2, 4, 4)
print(f"Weights sum: {weights.sum(dim=-1)}")  # All 1.0

# Test with mask
mask = torch.ones(batch, seq_len, seq_len)
mask[:, :, -1] = 0  # mask out last position
output_masked, weights_masked = scaled_dot_product_attention(Q, K, V, mask)
print(f"\nMasked weights (last column should be 0):")
print(weights_masked[0])
```
</details>

<details>
<summary><b>Solution 3: Why Scale by sqrt(d_k)?</b></summary>

1. **Variance of QK^T without scaling:** If Q and K have entries from N(0,1), then each entry of QK^T is a dot product of d_k pairs of independent N(0,1) variables. The variance of each product is 1, and the variance of their sum is **d_k**. So the standard deviation of scores is **sqrt(d_k)**.

2. **Softmax with large inputs:** When inputs are very large, softmax becomes very peaked -- it approaches a one-hot vector. In this regime:
   - The gradient of softmax becomes very small (near-zero)
   - Training effectively stalls because gradients vanish
   - The model becomes overconfident and cannot learn to attend to multiple positions

3. **How scaling fixes it:** Dividing by sqrt(d_k) normalizes the variance of scores back to approximately 1, regardless of d_k. This keeps the softmax in a well-behaved regime where gradients flow effectively. Without scaling, large d_k (like 512) would produce scores with std ~22.6, making softmax extremely peaked.

4. **Empirical results:**
   ```
   d_k= 64 | Unscaled std: 8.00  | Scaled std: 1.00
   d_k=256 | Unscaled std: 16.00 | Scaled std: 1.00
   d_k=512 | Unscaled std: 22.63 | Scaled std: 1.00
   ```
   Scaling keeps the standard deviation at ~1.0 regardless of dimension.
</details>

<details>
<summary><b>Solution 4: Positional Encoding Analysis</b></summary>

**1. Manual computation for d_model=4:**
```
PE(0) = [sin(0/1), cos(0/1), sin(0/100), cos(0/100)]
      = [sin(0),   cos(0),   sin(0),     cos(0)     ]
      = [0,        1,        0,          1           ]

PE(1) = [sin(1/1), cos(1/1), sin(1/100), cos(1/100)]
      = [sin(1),   cos(1),   sin(0.01),  cos(0.01)  ]
      = [0.8415,   0.5403,   0.0100,     0.9999     ]

PE(2) = [sin(2),   cos(2),   sin(0.02),  cos(0.02)  ]
      = [0.9093,   -0.4161,  0.0200,     0.9998     ]
```

**2. Dot product distances:** The dot product between PE(0) and PE(1) is larger than between PE(0) and PE(10), and much larger than between PE(0) and PE(100). Nearby positions have more similar encodings, which allows the model to learn relative position relationships.

**3. Why sin AND cos:** Using both allows the model to learn to attend to **relative positions**. For any fixed offset k, PE(pos+k) can be expressed as a linear function of PE(pos). Specifically, `PE(pos+k) = A_k * PE(pos)` for a rotation matrix A_k that depends only on k, not on pos. This means the model can learn "attend to the token 3 positions back" as a simple linear operation.

**4. Advantages over learned positional embeddings:**
- Can generalize to sequence lengths longer than those seen during training (extrapolation)
- No additional parameters to learn
- Smooth, continuous representation of position
- In practice, learned positional embeddings often work equally well or better for fixed-length tasks, but sinusoidal is more principled.
</details>

<details>
<summary><b>Solution 5: Transformer Architecture -- Component Inventory</b></summary>

**1. Multi-Head Attention:**
- **Purpose:** Allows each position to attend to all other positions, capturing dependencies regardless of distance. Multiple heads learn different types of relationships.
- **Without it:** The model has no mechanism for tokens to exchange information. It would process each token independently, like a bag-of-words model.

**2. Residual (Skip) Connection:**
- **Purpose:** Adds the input directly to the output of each sublayer: `output = sublayer(x) + x`. Enables gradient flow through deep networks.
- **Without it:** Gradients must flow through every sublayer sequentially. In deep transformers (6+ layers), this leads to vanishing gradients and makes training much harder.

**3. Layer Normalization:**
- **Purpose:** Normalizes activations across the feature dimension (not batch dimension). Stabilizes training by keeping activation distributions consistent.
- **Without it:** Activations can drift to very large or small values, making training unstable, especially with residual connections.

**4. Feed-Forward Network (FFN):**
- **Purpose:** Applies a non-linear transformation to each position independently. Adds representational capacity beyond what attention provides.
- **Without it:** The model would be limited to linear operations plus attention (which is itself a weighted linear combination). The FFN introduces the non-linearity needed for complex function approximation.

**5. Dropout:**
- **Purpose:** Regularization -- randomly zeroes elements to prevent overfitting.
- **Without it:** Transformers are large models that can easily overfit, especially on small datasets.

**FFN inner dimension:** Typically **4x d_model** (e.g., d_model=512, d_ff=2048). This expansion creates a "bottleneck" architecture that first expands to a higher dimension (with non-linearity) then projects back down.

**Why two layers with non-linearity:** A single large linear layer computes a linear transformation, no matter how wide. Two linear layers with a non-linearity (ReLU/GELU) between them can approximate any function (universal approximation). The intermediate non-linearity is essential for learning complex patterns.
</details>

<details>
<summary><b>Solution 6: Masking in Transformers</b></summary>

```python
import torch

# Padding mask
sequences = torch.tensor([
    [5, 12, 3, 0, 0],
    [8, 2, 6, 7, 0],
])

def create_padding_mask(sequences, pad_idx=0):
    """Create padding mask: 1 for real tokens, 0 for padding."""
    mask = (sequences != pad_idx).float()  # (batch, seq_len)
    return mask.unsqueeze(1).unsqueeze(2)  # (batch, 1, 1, seq_len)

pad_mask = create_padding_mask(sequences)
# Seq 0: [1, 1, 1, 0, 0]
# Seq 1: [1, 1, 1, 1, 0]

# Causal mask
def create_causal_mask(seq_len):
    """Create lower-triangular causal mask."""
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask.unsqueeze(0).unsqueeze(0)  # (1, 1, seq_len, seq_len)

causal_mask = create_causal_mask(5)
# [[1, 0, 0, 0, 0],
#  [1, 1, 0, 0, 0],
#  [1, 1, 1, 0, 0],
#  [1, 1, 1, 1, 0],
#  [1, 1, 1, 1, 1]]

# Combined mask (for decoder with padding):
# combined = causal_mask * pad_mask  # element-wise AND
```

**How masks work with attention:**
- Where mask = 0, we set the attention score to -infinity before softmax
- After softmax, these positions get weight ~0, effectively ignoring them
- The padding mask ensures padding tokens do not contribute to the output
- The causal mask ensures position i only attends to positions 0..i (no future leaking)
</details>

<details>
<summary><b>Solution 7: Multi-Head Attention -- Shape Tracking</b></summary>

**Setup:** `d_model=512, num_heads=8, d_k=64, batch=4, seq_len=20`

**Shape trace:**

```
Step                          | Shape
------------------------------|----------------------------
Input x                       | (4, 20, 512)
After Q = W_q(x)              | (4, 20, 512)
After K = W_k(x)              | (4, 20, 512)
After V = W_v(x)              | (4, 20, 512)
Split Q into heads            | (4, 8, 20, 64)
Split K into heads            | (4, 8, 20, 64)
Split V into heads            | (4, 8, 20, 64)
Attention scores (Q @ K^T)    | (4, 8, 20, 20)
Attention weights (softmax)   | (4, 8, 20, 20)
Attention output (weights @ V)| (4, 8, 20, 64)
Concatenate heads             | (4, 20, 512)
Output projection W_o         | (4, 20, 512)
```

**Parameter count (no biases):**
- W_q: 512 * 512 = 262,144
- W_k: 512 * 512 = 262,144
- W_v: 512 * 512 = 262,144
- W_o: 512 * 512 = 262,144
- **Total: 4 * 512^2 = 1,048,576** (~1M parameters)

**Single head comparison:** With a single head using d_k=512, we would still need W_q, W_k, W_v, W_o of the same dimensions (512x512 each), so the parameter count would be **identical**: 4 * 512^2 = 1,048,576.

Multi-head attention does NOT add parameters compared to single-head -- it simply partitions the same parameters into independent subspaces. The advantage is that different heads can learn different attention patterns (e.g., one head attends to adjacent tokens, another to syntactic dependencies), which a single head cannot do simultaneously.
</details>